In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN,Dense

In [3]:
train=pd.read_csv("train.txt", sep=";", names=["text", "emotion"])
test=pd.read_csv("test.txt", sep=";", names=["text", "emotion"])
val=pd.read_csv("val.txt", sep=";", names=["text", "emotion"])

emotions=["sadness", "joy", "love", "anger", "fear", "surprise"]

for data in [train, test, val]:
    data["emotion"] = data["emotion"].str.strip().str.lower()
    data["label"] = data["emotion"].apply(emotions.index)

print("Dataset Loaded Successfully!")
print(train.head())

Dataset Loaded Successfully!
                                                text  emotion  label
0                            i didnt feel humiliated  sadness      0
1  i can go from feeling so hopeless to so damned...  sadness      0
2   im grabbing a minute to post i feel greedy wrong    anger      3
3  i am ever feeling nostalgic about the fireplac...     love      2
4                               i am feeling grouchy    anger      3


In [4]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(train["text"])
X_train = pad_sequences(
    tokenizer.texts_to_sequences(train["text"]), maxlen=40
)

X_test = pad_sequences(
    tokenizer.texts_to_sequences(test["text"]), maxlen=40
)

X_val = pad_sequences(
    tokenizer.texts_to_sequences(val["text"]), maxlen=40
)

y_train = train["label"]
y_test = test["label"]
y_val = val["label"]

In [5]:
model = Sequential([
    Embedding(5000, 32),
    SimpleRNN(32),
    Dense(6, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [12]:
model.fit(X_train, y_train, epochs=30,validation_data=(X_val, y_val))

Epoch 1/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9909 - loss: 0.0366 - val_accuracy: 0.7360 - val_loss: 1.3514
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9823 - loss: 0.0555 - val_accuracy: 0.7420 - val_loss: 1.3248
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9896 - loss: 0.0357 - val_accuracy: 0.7440 - val_loss: 1.4101
Epoch 4/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9584 - loss: 0.1434 - val_accuracy: 0.7230 - val_loss: 1.3987
Epoch 5/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9851 - loss: 0.0530 - val_accuracy: 0.7245 - val_loss: 1.4276
Epoch 6/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9926 - loss: 0.0286 - val_accuracy: 0.7185 - val_loss: 1.4218
Epoch 7/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9943 - loss: 0.0179 - val_accuracy: 0.7480 - val_loss: 1.3297
Epoch 8/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9963 - loss: 0.0120 - val_accuracy: 0.

In [13]:
loss,accuracy=model.evaluate(X_test,y_test)
print("Test Loss:",loss)
print("Test Accuracy:",accuracy*100)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7265 - loss: 1.5523
Test Loss: 1.552290916442871
Test Accuracy: 72.64999747276306


In [15]:
def predict_emotion():
    text = input("Enter your sentence: ")
    x = tokenizer.texts_to_sequences([text])
    x = pad_sequences(x, maxlen=40)
    prediction = model.predict(x, verbose=0)
    result = emotions[np.argmax(prediction)]
    print("Your Input:", text)
    print("Predicted Emotion:", result.upper())
predict_emotion()

Enter your sentence: I AM VERY HAPPY  TODAY
Your Input: I AM VERY HAPPY  TODAY
Predicted Emotion: JOY
